In [2]:
import torch
from torch import nn

In [3]:
# detecter et utiliser l'accelerateur pour votre machine
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [4]:
# Definir le model
class NeuralNet(nn.Module):
    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5),  # (N,  6, 28, 28)
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),                    # (N,  6, 14, 14)
 
            nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5), # (N, 16, 10, 10)
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),                    # (N, 16,  5,  5)
        )
 
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 5 * 5, 120),
            nn.Tanh(),
            nn.Linear(120, 84),
            nn.Tanh(),
            nn.Linear(84, num_classes),   # raw logits
        )
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.feature_extractor(x)
        x = self.classifier(x)
        return x

In [5]:
model = NeuralNet(num_classes=10)
print(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {n_params:,}")

dummy = torch.randn(8, 1, 32, 32)
logits = model(dummy)
print(f"Input  shape : {dummy.shape}")
print(f"Output shape : {logits.shape}")
print(f"Predictions  : {logits.argmax(dim=1).tolist()}")

NeuralNet(
  (feature_extractor): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
    (1): Tanh()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (4): Tanh()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=400, out_features=120, bias=True)
    (2): Tanh()
    (3): Linear(in_features=120, out_features=84, bias=True)
    (4): Tanh()
    (5): Linear(in_features=84, out_features=10, bias=True)
  )
)

Trainable parameters: 61,706
Input  shape : torch.Size([8, 1, 32, 32])
Output shape : torch.Size([8, 10])
Predictions  : [6, 6, 1, 6, 1, 1, 6, 1]


In [ ]:
# Loader et préparer les données 


In [ ]:
# Definir les hyperparamètres et effectuer l'entrainement